#Llave
esta llave tiene vigiencia de 7 dias

In [ ]:
# 1. Definir la variable
mi_llave_api = "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"


#Codigo del bot
este es el codigo para utilizar en la pagina.
* Ya tiene asignado el rol
* trabaja con ia generativa
* permite multiples usuarios para


In [ ]:
# 1. Instalamos la librería
!pip install groq -q

from groq import Groq

# 2. Configuración (Aquí pones tu llave)
client = Groq(api_key=mi_llave_api)

# 2. Definición del ROL MAESTRO (Fuera de la función para mayor orden)
ROL_MAESTRO_SOLAR = """
Eres el 'Asistente Solar', una IA servicial y experta, dedicada a educar sobre sistemas solares en Zonas Rurales y No Interconectadas (ZNI).

PERSONALIDAD Y DISCRECIÓN:
1. No interrogues al usuario. Saluda de forma acogedora y preséntate como un guía para soluciones de energía en el campo o lugares alejados.
2. Si el usuario no dice dónde vive, mantén un lenguaje neutral-rural (ej. "Ideal para fincas o casas de campo"). No presiones por su ubicación.
3. Si el usuario menciona que es de la CIUDAD, redirige con suavidad: "¡Qué bien! Aunque mi especialidad es apoyar a quienes están lejos de la red eléctrica, puedo contarte cómo estas soluciones cambian la vida en el campo o ayudarte si tienes algún proyecto para una zona rural".

MISION EDUCATIVA (Prioridades del Reto):
- BENEFICIOS: Explica cómo la energía solar permite estudiar de noche, refrigerar medicinas/alimentos y bombear agua.
- TÉCNICO: Enseña sobre la cadena Panel -> Regulador -> Batería -> Inversor de forma sencilla (usa analogías como "el tanque de agua").
- MANTENIMIENTO: Da consejos prácticos como limpiar el polvo de los paneles o cuidar las baterías del calor.

ESTILO DE RESPUESTA:
- Sencillo y práctico.
- Evita párrafos largos a menos que el usuario pida profundizar.
- Siempre prioriza la seguridad eléctrica en tus consejos.
"""
historiales = {} # Diccionario global

def chatbot_academico(user_id, mensaje_nuevo):
    # Inicialización del usuario con el nuevo Rol Maestro
    if user_id not in historiales or isinstance(historiales[user_id], list):
        historiales[user_id] = {
            "mensajes": [
                {"role": "system", "content": ROL_MAESTRO_SOLAR}
            ],
            "nivel_interes": 1
        }

    sesion = historiales[user_id]

    # Lógica de detección de interés (Sube nivel si detecta curiosidad)
    palabras_interes = ["detalla", "explica más", "técnico", "cómo funciona", "profundiza", "por qué", "interesante", "especificaciones"]
    if any(palabra in mensaje_nuevo.lower() for palabra in palabras_interes):
        sesion["nivel_interes"] = min(sesion["nivel_interes"] + 1, 5)

    # Ajustes dinámicos de personalidad
    ajustes = {
        1: {"temp": 0.7, "tokens": 150, "estilo": "muy breve, motivador y cercano"},
        2: {"temp": 0.6, "tokens": 250, "estilo": "informativo, usando analogías simples"},
        3: {"temp": 0.5, "tokens": 400, "estilo": "detallado y con ejemplos claros"},
        4: {"temp": 0.4, "tokens": 600, "estilo": "técnico, preciso y educativo"},
        5: {"temp": 0.2, "tokens": 800, "estilo": "experto avanzado, exhaustivo y técnico"}
    }

    config = ajustes[sesion["nivel_interes"]]

    # Instrucción de estilo que se suma al Rol Maestro en cada respuesta
    instruccion_estilo = f"\n[INSTRUCCIÓN DE ESTILO ACTUAL]: Responde de forma {config['estilo']}. Si la respuesta es corta por naturaleza, NO la alargues innecesariamente."

    # Guardamos el mensaje del usuario
    sesion["mensajes"].append({"role": "user", "content": mensaje_nuevo})

    # Llamada a la IA (Combinamos el historial con la instrucción de estilo actual)
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=sesion["mensajes"] + [{"role": "system", "content": instruccion_estilo}],
        temperature=config["temp"],
        max_tokens=config["tokens"]
    )

    respuesta_bot = response.choices[0].message.content

    # Guardamos la respuesta en el historial para que el bot tenga memoria
    sesion["mensajes"].append({"role": "assistant", "content": respuesta_bot})

    return f"[Nivel de Interés: {sesion['nivel_interes']}/5]\nBot: {respuesta_bot}"

#Prueba
Solo es para un usuario.

Pueden ponerlo a prueba pára revisar si es lo que consideran que el reto solicita

In [ ]:
# --- MODO CHAT INTERACTIVO ---

print("--- Bienvenido al Chatbot Académico ---")
print("(Escribe 'salir' para terminar la conversación)")

# Definimos quién eres para esta prueba
mi_usuario = "Estudiante_Prueba"

while True:
    # 1. El programa espera a que tú escribas algo
    mensaje = input("Tú: ")

    # 2. Si escribes 'salir', se rompe el bucle
    if mensaje.lower() in ["salir", "exit", "chau"]:
        print("Bot: ¡Hasta luego! Mucho éxito en tus estudios.")
        break

    # 3. Llamamos a la función que ya creamos
    respuesta = chatbot_academico(mi_usuario, mensaje)

    # 4. Mostramos la respuesta del bot
    print(f"Bot: {respuesta}")
    print("-" * 30)